# YouTube Playlist Processing Pipeline - Google Colab

This notebook helps you run the YouTube playlist processing pipeline on Google Colab with GPU acceleration for faster transcription and audio processing.

## Features
- Download entire YouTube playlists
- Convert to MP3
- Remove noise using DeepFilterNet
- Normalize audio volume
- Transcribe using Faster-Whisper (GPU accelerated)
- Summarize using OpenAI GPT

## Step 1: Check GPU Availability

First, let's check if a GPU is available for faster processing.

In [ ]:
!nvidia-smi

## Step 2: Clone the Repository

In [ ]:
!git clone https://github.com/creative-h/noiseFilterYT.git
%cd noiseFilterYT

## Step 3: Install Dependencies

This will install all required packages including FFmpeg, yt-dlp, DeepFilterNet, and Faster-Whisper.

In [ ]:
# Install FFmpeg
!apt-get update -qq
!apt-get install -y ffmpeg

# Install Python dependencies
!pip install -q yt-dlp deepfilternet faster-whisper openai

# Install CUDA-compatible PyTorch for GPU acceleration
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu118

## Step 4: Configure OpenAI API (Optional)

If you want to use the summarization feature, set your OpenAI API key here.

In [ ]:
import os

# Enter your OpenAI API key here (or leave empty to skip summarization)
OPENAI_API_KEY = ""  # @param {type:"string"}

if OPENAI_API_KEY:
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    print("OpenAI API key set")
else:
    print("No API key set - summarization will be skipped")

## Step 5: Configure Pipeline Settings

Adjust these settings based on your needs.

In [ ]:
# Playlist URL
PLAYLIST_URL = ""  # @param {type:"string"}

# Processing options
START_INDEX = 1  # @param {type:"integer"}
END_INDEX = None  # @param {type:"integer"}

# Skip steps (set to True to skip)
SKIP_DENOISE = False  # @param {type:"boolean"}
SKIP_NORMALIZE = False  # @param {type:"boolean"}
SKIP_TRANSCRIBE = False  # @param {type:"boolean"}
SKIP_SUMMARIZE = False  # @param {type:"boolean"}

# Whisper model (tiny, base, small, medium, large-v3)
# Use 'distil-large-v3' for faster transcription with good accuracy
WHISPER_MODEL = "large-v3"  # @param ["tiny", "base", "small", "medium", "large-v3", "distil-large-v3"]

# Device (auto, cpu, cuda)
WHISPER_DEVICE = "auto"  # @param ["auto", "cpu", "cuda"]

print(f"Playlist URL: {PLAYLIST_URL}")
print(f"Whisper Model: {WHISPER_MODEL}")
print(f"Device: {WHISPER_DEVICE}")

## Step 6: Update Configuration

In [ ]:
# Update config.py with our settings
config_content = '''
"""
Configuration settings for the YouTube playlist processing pipeline.
"""

import os
from pathlib import Path

# Base directory
BASE_DIR = Path(__file__).parent

# Directory structure
RAW_DIR = BASE_DIR / "raw"
MP3_DIR = BASE_DIR / "mp3"
CLEAN_DIR = BASE_DIR / "clean"
TRANSCRIPT_DIR = BASE_DIR / "transcript"
SUMMARY_DIR = BASE_DIR / "summary"
LOGS_DIR = BASE_DIR / "logs"

# Create directories if they don't exist
for dir_path in [RAW_DIR, MP3_DIR, CLEAN_DIR, TRANSCRIPT_DIR, SUMMARY_DIR, LOGS_DIR]:
    dir_path.mkdir(parents=True, exist_ok=True)

# yt-dlp settings
YDLP_FORMAT = "bestaudio/best"
YDLP_POSTPROCESSORS = [
    {
        "key": "FFmpegExtractAudio",
        "preferredcodec": "mp3",
        "preferredquality": "192",
    }
]

# Audio processing settings
AUDIO_BITRATE = "192k"
AUDIO_SAMPLE_RATE = 44100

# FFmpeg loudnorm settings
LOUDNORM_SETTINGS = {
    "i": "-16",
    "tp": "-1.5",
    "lra": "11",
}

# DeepFilterNet settings
DEEPFILTER_MODEL = "deepfilter_net"

# Whisper settings
WHISPER_MODEL = "''' + WHISPER_MODEL + '''"
WHISPER_DEVICE = "''' + WHISPER_DEVICE + '''"
WHISPER_COMPUTE_TYPE = "float16"

# LLM settings for summarization
LLM_API_KEY = os.getenv("OPENAI_API_KEY", "")
LLM_MODEL = "gpt-4o"
LLM_TEMPERATURE = 0.7
LLM_MAX_TOKENS = 2000

# Logging settings
LOG_LEVEL = "INFO"
LOG_FORMAT = "%(asctime)s - %(name)s - %(levelname)s - %(message)s"
LOG_FILE = LOGS_DIR / "pipeline.log"

# Processing options
SKIP_EXISTING = True
REMOVE_SILENCE = False
'''

with open('config.py', 'w') as f:
    f.write(config_content)

print("Configuration updated")

## Step 7: Run the Pipeline

This will download and process the entire playlist (or the specified range).

In [ ]:
from main import Pipeline

# Initialize pipeline
pipeline = Pipeline()

# Run full pipeline
pipeline.run_full_pipeline(
    playlist_url=PLAYLIST_URL,
    start=START_INDEX if START_INDEX > 0 else None,
    end=END_INDEX if END_INDEX and END_INDEX > 0 else None,
    skip_denoise=SKIP_DENOISE,
    skip_normalize=SKIP_NORMALIZE,
    skip_transcribe=SKIP_TRANSCRIBE,
    skip_summarize=SKIP_SUMMARIZE,
)

## Step 8: Download Results

After processing, you can download the results.

In [ ]:
# List all processed files
import os
from pathlib import Path

print("=== Processed Files ===")
print(f"\nRaw audio files: {len(list(Path('raw').glob('*')))}")
print(f"MP3 files: {len(list(Path('mp3').glob('*.mp3')))}")
print(f"Clean audio files: {len(list(Path('clean').glob('*.mp3')))}")
print(f"Transcripts: {len(list(Path('transcript').glob('*.txt')))}")
print(f"Summaries: {len(list(Path('summary').glob('*.md')))}")

### Download Individual Files

Run this cell to see download links for all processed files.

In [ ]:
from google.colab import files

# Download all transcripts
transcript_files = list(Path('transcript').glob('*.txt'))
if transcript_files:
    print(f"Downloading {len(transcript_files)} transcript files...")
    for f in transcript_files:
        files.download(f)
else:
    print("No transcript files found")

### Download as ZIP

Download all results as a single ZIP file.

In [ ]:
# Create ZIP of all results
!zip -r results.zip clean/ transcript/ summary/

# Download ZIP
from google.colab import files
files.download('results.zip')

## Alternative: Process Single Video

If you want to process just one video instead of a playlist.

In [ ]:
# Single video URL
VIDEO_URL = ""  # @param {type:"string"}

if VIDEO_URL:
    pipeline.process_single_video(
        video_url=VIDEO_URL,
        skip_denoise=SKIP_DENOISE,
        skip_normalize=SKIP_NORMALIZE,
        skip_transcribe=SKIP_TRANSCRIBE,
        skip_summarize=SKIP_SUMMARIZE,
    )
else:
    print("Please enter a video URL")

## Tips for Large Playlists

1. **Process in chunks**: Use START_INDEX and END_INDEX to process playlists in batches
2. **Use faster model**: Set WHISPER_MODEL to "distil-large-v3" for faster transcription
3. **Skip steps**: If you only need transcripts, set SKIP_DENOISE and SKIP_NORMALIZE to True
4. **Save to Drive**: Mount Google Drive to save results permanently

### Mount Google Drive (Optional)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy results to Drive
!cp -r clean/ /content/drive/MyDrive/youtube_pipeline/
!cp -r transcript/ /content/drive/MyDrive/youtube_pipeline/
!cp -r summary/ /content/drive/MyDrive/youtube_pipeline/

## Troubleshooting

- **Out of memory**: Use a smaller Whisper model ("medium" or "small")
- **Slow processing**: Ensure GPU is enabled in Colab (Runtime > Change runtime type > GPU)
- **Download errors**: Check playlist URL is public and accessible
- **DeepFilterNet errors**: Skip denoising with SKIP_DENOISE=True